# Técnicas de Diseño y Validación de Modelos · Universidad Blas Pascal
## Módulo 5 · Optimización de Modelos e Hiperparámetros
## Laboratorio: Grid Search vs. Random Search vs. Optuna (TPE)

<div style="display:flex; justify-content:flex-start; margin: 8px 0 18px 0;">
  <a href="https://colab.research.google.com/github/GabrielaOjcius/Laboratorios-Tecnicas-Diseno-Validacion-Modelos/blob/main/Laboratorios%20opcionales/M5_Notebook.ipynb" target="_blank">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Abrir en Colab"/>
  </a>
  <span style="font-size: 0.98em;">En Colab, hac&eacute; una copia en tu Drive antes de editar.</span>
</div>


---

### Objetivo
Comparar empíricamente las tres estrategias de HPO sobre el **dataset Wisconsin Breast Cancer Diagnostic** usando `GradientBoostingClassifier`. Se miden tres dimensiones: mejor AUC en CV, AUC final en test y tiempo de búsqueda. El experimento también demuestra el protocolo de evaluación honesta completo y el concepto de sesgo de optimismo.

### Alineación con el material teórico
Corresponde a la **Actividad Optativa** de `Módulo 5 · Optimización de Modelos e Hiperparámetros`. Prepara las **Secciones C y D** de la Parte 2 de la Evaluación Integradora.

### Protocolo de cinco pasos
1. Particionar train/test **antes** de cualquier análisis ← esta celda
2. Diagnóstico sobre train (curvas de aprendizaje del modelo base)
3. Seleccionar estrategia de búsqueda
4. HPO sobre **train** con CV interna
5. Evaluación final **una sola vez** sobre test

In [ ]:
# ─── Instalación de Optuna (solo en Colab) ────────────────────────────────────
!pip install optuna --quiet

import time
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import (
    train_test_split, StratifiedKFold,
    GridSearchCV, RandomizedSearchCV,
    cross_val_score
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score, classification_report
from scipy.stats import randint, uniform
import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

print('Optuna version:', optuna.__version__)

In [ ]:
# ─── B.1. PASO 1: Particionar train/test antes de cualquier análisis ──────────
X, y = load_breast_cancer(return_X_y=True)

# random_state=42 fijado para reproducibilidad
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.20, stratify=y, random_state=42
)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def construir_pipe():
    return Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    GradientBoostingClassifier(random_state=42))
    ])

print(f'Train: {X_tr.shape}  |  Test: {X_te.shape}')
print(f'Proporción positivos train: {y_tr.mean():.3f}  test: {y_te.mean():.3f}')
print('Test reservado. No se toca hasta la evaluación final (B.6).')

In [ ]:
# ─── B.2. PASO 4a: Grid Search exhaustivo ─────────────────────────────────────
param_grid = {
    'clf__n_estimators':  [50, 150, 250],    # 3
    'clf__max_depth':     [2, 4, 6],          # 3
    'clf__learning_rate': [0.01, 0.05, 0.20], # 3
    'clf__subsample':     [0.6, 0.8, 1.0],    # 3
}  # Total: 81 combinaciones × 5 folds = 405 entrenamientos

t0 = time.time()
grid = GridSearchCV(
    construir_pipe(), param_grid,
    cv=cv, scoring='roc_auc', n_jobs=-1, return_train_score=False
)
grid.fit(X_tr, y_tr)
t_grid = time.time() - t0

print('=== GRID SEARCH ===')
print(f'Combinaciones evaluadas: {len(grid.cv_results_["mean_test_score"])}')
print(f'Mejor AUC CV:            {grid.best_score_:.4f}')
print(f'Tiempo:                  {t_grid:.1f} s')
print(f'Mejores HP:              {grid.best_params_}')

In [ ]:
# ─── B.3. PASO 4b: Random Search con 30 trials ────────────────────────────────
param_dist = {
    'clf__n_estimators':  randint(50, 300),
    'clf__max_depth':     randint(2, 7),
    'clf__learning_rate': uniform(0.005, 0.245),  # [0.005, 0.25]
    'clf__subsample':     uniform(0.5, 0.5),       # [0.5, 1.0]
}

t0 = time.time()
rs = RandomizedSearchCV(
    construir_pipe(), param_dist, n_iter=30,
    cv=cv, scoring='roc_auc', n_jobs=-1, random_state=42
)
rs.fit(X_tr, y_tr)
t_rs = time.time() - t0

print('=== RANDOM SEARCH (30 trials) ===')
print(f'Mejor AUC CV:  {rs.best_score_:.4f}')
print(f'Tiempo:        {t_rs:.1f} s')
print(f'Mejores HP:    {rs.best_params_}')

# Evolución del mejor AUC por número de trial
best_so_far_rs = np.maximum.accumulate(rs.cv_results_['mean_test_score'])

In [ ]:
# ─── B.4. PASO 4c: Optuna con TPE sampler (30 trials) ────────────────────────
def objective(trial):
    params = {
        'n_estimators':  trial.suggest_int('n_estimators', 50, 300),
        'max_depth':     trial.suggest_int('max_depth', 2, 6),
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.25, log=True),
        'subsample':     trial.suggest_float('subsample', 0.5, 1.0),
    }
    pipe = Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    GradientBoostingClassifier(**params, random_state=42))
    ])
    return cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring='roc_auc', n_jobs=-1).mean()

t0 = time.time()
study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=42)
)
study.optimize(objective, n_trials=30, show_progress_bar=False)
t_opt = time.time() - t0

print('=== OPTUNA TPE (30 trials) ===')
print(f'Mejor AUC CV:  {study.best_value:.4f}')
print(f'Tiempo:        {t_opt:.1f} s')
print(f'Mejores HP:    {study.best_params}')

scores_opt = np.array([t.value for t in study.trials])
best_so_far_opt = np.maximum.accumulate(scores_opt)

In [ ]:
# ─── B.5. Curva de convergencia: Random Search vs. Optuna ────────────────────
fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(range(1, 31), best_so_far_rs,  '-o', color='steelblue', lw=2, label='Random Search')
ax.plot(range(1, 31), best_so_far_opt, '-o', color='#8B1A2F', lw=2, label='Optuna (TPE)')
ax.axhline(grid.best_score_, color='gray', ls='--', lw=1.5,
           label=f'Grid Search (referencia): {grid.best_score_:.4f}')
ax.set_xlabel('Número de trial')
ax.set_ylabel('Mejor AUC CV encontrado hasta ese trial')
ax.set_title('Convergencia de la búsqueda de hiperparámetros')
ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

print('\n=== RESUMEN COMPARATIVO ===')
print(f'Grid Search    | AUC CV: {grid.best_score_:.4f} | Tiempo: {t_grid:.1f}s | Eval: 81')
print(f'Random Search  | AUC CV: {rs.best_score_:.4f} | Tiempo: {t_rs:.1f}s | Eval: 30')
print(f'Optuna TPE     | AUC CV: {study.best_value:.4f} | Tiempo: {t_opt:.1f}s | Eval: 30')

In [ ]:
# ─── B.6. PASO 5: Evaluación final honesta ────────────────────────────────────
# Reentrenar el mejor modelo en TODO el train (sin CV)
# Evaluar UNA SOLA VEZ sobre test reservado

mejores = {
    'Grid Search':   grid.best_estimator_,
    'Random Search': rs.best_estimator_,
    'Optuna TPE':    Pipeline([
        ('scaler', StandardScaler()),
        ('clf',    GradientBoostingClassifier(**study.best_params, random_state=42))
    ]).fit(X_tr, y_tr)
}

print(f"{'Estrategia':14} | {'AUC CV':>8} | {'AUC TEST':>8} | {'Sesgo opt.':>10}")
print('-' * 52)
for nombre, modelo in mejores.items():
    auc_cv = {'Grid Search': grid.best_score_,
               'Random Search': rs.best_score_,
               'Optuna TPE': study.best_value}[nombre]
    proba_te = modelo.predict_proba(X_te)[:, 1]
    auc_te   = roc_auc_score(y_te, proba_te)
    print(f'{nombre:14} | {auc_cv:>8.4f} | {auc_te:>8.4f} | {auc_cv-auc_te:>+10.4f}')

print('\nNota: el sesgo de optimismo (AUC_CV − AUC_test) es positivo por construcción.')

In [ ]:
# ─── B.7. Validación cruzada anidada (estimación insesgada del proceso) ───────
cv_int = StratifiedKFold(n_splits=4, shuffle=True, random_state=1)
cv_ext = StratifiedKFold(n_splits=5, shuffle=True, random_state=2)

# El bucle INTERNO hace la búsqueda de HP
busqueda = RandomizedSearchCV(
    construir_pipe(), param_dist, n_iter=15,
    cv=cv_int, scoring='roc_auc', n_jobs=-1, random_state=42
)

# El bucle EXTERNO evalúa el proceso completo
t0 = time.time()
auc_nested = cross_val_score(
    busqueda, X_tr, y_tr, cv=cv_ext, scoring='roc_auc', n_jobs=-1
)
t_nested = time.time() - t0

print('=== VALIDACIÓN CRUZADA ANIDADA ===')
print(f'AUC anidado (insesgado): {auc_nested.mean():.4f} ± {auc_nested.std():.4f}')
print(f'Tiempo total:            {t_nested:.1f} s')
print(f'\nComparacion con RS no anidado: {rs.best_score_:.4f}')
print(f'El AUC anidado mide el proceso completo; suele ser menor que el CV no anidado.')

In [ ]:
# ─── B.8. Importancia de hiperparámetros (Optuna) ────────────────────────────
import optuna.importance as oi
importancias = oi.get_param_importances(study)
print('Importancia relativa de hiperparámetros:')
for hp, imp in importancias.items():
    barra = '#' * int(imp * 40)
    print(f'  {hp:20s} {barra:<40} {imp:.3f}')
print()
dominante = max(importancias, key=importancias.get)
print(f'HP dominante: {dominante} ({importancias[dominante]:.1%} de la varianza del score)')
print('Recomendación: refinar el rango de búsqueda de ese HP en una segunda búsqueda.')

---
## Preguntas de análisis
Responder en las celdas markdown siguientes. Hacer referencia a los valores numéricos de las tablas de B.5, B.6 y B.7.

### Pregunta 1
¿Optuna converge más rápido que Random Search? Identificar el número de trial aproximado en que Optuna alcanza el 99 % de su AUC final. Comparar con Random Search.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 2
¿La mejor configuración encontrada coincide entre las tres estrategias? Si difieren en algún hiperparámetro, ¿significa que una estrategia es mejor? Razonar sobre el ruido estadístico y los múltiples óptimos locales.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 3
Comparar el AUC de CV de cada estrategia con el AUC en test (tabla de B.6). ¿En qué dirección es el sesgo? ¿Por qué el AUC de CV siempre sobreestima el AUC real? Conectar con el concepto de sesgo de optimismo.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 4
El AUC de Nested CV (B.7) es menor que el AUC de CV no anidado de Random Search. Explicar por qué esta diferencia es esperada y qué mide exactamente cada estimación.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`

### Pregunta 5
¿En qué situaciones de trabajo profesional preferiría cada una de las tres estrategias (Grid Search, Random Search, Optuna)? Justificar con criterios de tiempo, presupuesto computacional y conocimiento previo del espacio.

**Respuesta:**

`# ✏️ RESPONDA AQUÍ`